In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
from matplotlib.colors import ListedColormap

In [ ]:
metvar = "tp"

In [ ]:
nysm_radios = pd.read_csv(
    "/home/aevans/nwp_bias/src/machine_learning/notebooks/data/radiometer_network_nysm_stations.csv"
)
stations = nysm_radios["stid"].unique()

In [ ]:
main_path = "/home/aevans/nwp_bias/src/machine_learning/data/hybrid_output"

In [ ]:
dirs = [d for d in os.listdir(main_path) if d in stations]

hybrid_master = []
lstm_master = []
pers_master = []

# hybrid
for d in dirs:
    files = os.listdir(f"{master_dir}/{d}")
    files = [f for f in files if "linear" in f and "hybrid" in f]
    files = [f for f in files if metvar in f]

    for f in files:
        temp_ = pd.read_parquet(f"{master_dir}/{d}/{f}")
        hybrid_master.append(temp_)

# lstm
for d in dirs:
    files = os.listdir(f"{master_dir}/{d}")
    files = [f for f in files if "linear" in f and "lstm" in f]
    files = [f for f in files if metvar in f]

    for f in files:
        temp_ = pd.read_parquet(f"{master_dir}/{d}/{f}")
        lstm_master.append(temp_)


# persistence
for d in dirs:
    files = os.listdir(f"{master_dir}/{d}")
    files = [f for f in files if "linear" in f and "persistence" in f]
    files = [f for f in files if metvar in f]

    for f in files:
        temp_ = pd.read_parquet(f"{master_dir}/{d}/{f}")
        pers_master.append(temp_)


# ls --> dataframes
hybrid_mse = (
    pd.concat(hybrid_master, ignore_index=True)
    .sort_values(["station", "fh"])
    .reset_index(drop=True)
)

lstm_mse = (
    pd.concat(lstm_master, ignore_index=True)
    .sort_values(["station", "fh"])
    .reset_index(drop=True)
)

pers_mse = (
    pd.concat(pers_master, ignore_index=True)
    .sort_values(["station", "fh"])
    .reset_index(drop=True)
)

# plot mse and r^2

In [ ]:
stations = sorted(
    set(hybrid_mse["station"]) | set(lstm_mse["station"]) | set(pers_mse["station"])
)

cmap = plt.cm.tab20
station_colors = {station: cmap(i % cmap.N) for i, station in enumerate(stations)}

In [ ]:
def plot_station_skill_3panel(
    hybrid_df, lstm_df, pers_df, fh_range=(1, 19), figsize=(14, 12), cmap=plt.cm.tab20
):
    """
    Create a 3-panel figure showing MSE (circles) and R^2 (x's)
    vs forecast hour for each station, with consistent colors
    across Hybrid, LSTM, and Persistence models.

    Expected columns in each DataFrame:
        ['station', 'fh', 'mse', 'r2']
    """

    # ------------------------------------------------------------------
    # Build consistent station → color mapping
    # ------------------------------------------------------------------
    stations = sorted(
        set(hybrid_df["station"]) | set(lstm_df["station"]) | set(pers_df["station"])
    )

    station_colors = {station: cmap(i % cmap.N) for i, station in enumerate(stations)}

    # ------------------------------------------------------------------
    # Internal plotting helper
    # ------------------------------------------------------------------
    def _plot_model(ax, df, title):
        ax_r2 = ax.twinx()

        for station in stations:
            sub = df[df["station"] == station].sort_values("fh")
            if sub.empty:
                continue

            color = station_colors[station]

            # MSE → circles
            ax.scatter(sub["fh"], sub["mse"], color=color, marker="o", s=40, alpha=0.85)

            # R^2 → x's
            ax_r2.scatter(
                sub["fh"], sub["r2"], color=color, marker="x", s=40, alpha=0.85
            )

        ax.set_title(title)
        ax.set_ylabel("MSE")
        ax_r2.set_ylabel(r"$R^2$")
        ax.set_xlim(*fh_range)
        ax.grid(True, alpha=0.3)

    # ------------------------------------------------------------------
    # Create figure
    # ------------------------------------------------------------------
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=figsize, sharex=True)

    _plot_model(axes[0], hybrid_df, "Hybrid Model Performance")
    _plot_model(axes[1], lstm_df, "LSTM Model Performance")
    _plot_model(axes[2], pers_df, "Persistence Model Performance")

    axes[-1].set_xlabel("Forecast Hour")

    # ------------------------------------------------------------------
    # Shared legend (station colors only)
    # ------------------------------------------------------------------
    legend_handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=station_colors[s], label=s)
        for s in stations
    ]

    fig.legend(
        legend_handles,
        stations,
        loc="upper center",
        ncol=min(len(stations), 6),
        frameon=False,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
plot_station_skill_3panel(hybrid_mse, lstm_mse, pers_mse)

# weatherBench

In [ ]:
def weatherBench_percent(
    pers_df,
    hybrid_df,
    fh_range=(1, 18),
    figsize=(14, 8),
    diverging_cmap="RdBu_r",
    grey_cmap="Greys",
):
    """
    Seaborn-based 2-panel heatmap figure:
      1) HRRR / Persistence MSE (greys)
      2) Hybrid % improvement vs HRRR (blue-red)

    Expected columns: ['station', 'fh', 'mse']
    """

    # --------------------------------------------------------------
    # Pivot to station x forecast-hour matrices
    # --------------------------------------------------------------
    def pivot(df):
        return (
            df.pivot(index="station", columns="fh", values="mse")
            .sort_index()
            .loc[:, fh_range[0] : fh_range[1]]
        )

    pers_mse = pivot(pers_df)
    hybrid_mse = pivot(hybrid_df)

    # --------------------------------------------------------------
    # Percent improvement relative to HRRR
    # --------------------------------------------------------------
    hybrid_imp = 100 * (pers_mse - hybrid_mse) / pers_mse

    vmax = np.nanmax(np.abs(hybrid_imp.values))

    # --------------------------------------------------------------
    # Plot
    # --------------------------------------------------------------
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=figsize, sharex=True)

    # ---------------- HRRR MSE ----------------
    sns.heatmap(
        pers_mse,
        ax=axes[0],
        cmap=grey_cmap,
        annot=True,
        fmt=".2f",
        cbar_kws={"label": "MSE"},
        linewidths=0.3,
        linecolor="white",
    )

    axes[0].set_title("HRRR (Persistence) MSE")
    axes[0].set_ylabel("Station")
    axes[0].set_xlabel("")

    # ---------------- Hybrid Improvement ----------------
    sns.heatmap(
        hybrid_imp,
        ax=axes[1],
        cmap=diverging_cmap,
        center=0,
        vmin=-vmax,
        vmax=vmax,
        annot=True,
        fmt=".1f",
        cbar_kws={"label": "% Improvement vs HRRR"},
        linewidths=0.3,
        linecolor="white",
    )

    axes[1].set_title("Hybrid % Improvement vs HRRR")
    axes[1].set_ylabel("Station")
    axes[1].set_xlabel("Forecast Hour")

    plt.tight_layout()
    plt.show()

In [ ]:
def weatherBench_raw(
    pers_df,
    hybrid_df,
    fh_range=(1, 18),
    figsize=(14, 8),
    diverging_cmap="RdBu_r",
    grey_cmap="Greys",
):
    """
    Seaborn-based 2-panel heatmap figure:
      1) HRRR / Persistence MSE (greys, annotated with MSE)
      2) Hybrid MSE annotated, but colored by ΔMSE (Hybrid − HRRR)

    Expected columns: ['station', 'fh', 'mse']
    """

    # --------------------------------------------------------------
    # Pivot to station x forecast-hour matrices
    # --------------------------------------------------------------
    def pivot(df):
        return (
            df.pivot(index="station", columns="fh", values="mse")
            .sort_index()
            .loc[:, fh_range[0] : fh_range[1]]
        )

    pers_mse = pivot(pers_df)
    hybrid_mse = pivot(hybrid_df)

    # --------------------------------------------------------------
    # Raw difference for coloring
    # --------------------------------------------------------------
    diff_mse = hybrid_mse - pers_mse

    vmax = np.nanmax(np.abs(diff_mse.values))

    # --------------------------------------------------------------
    # Plot
    # --------------------------------------------------------------
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=figsize, sharex=True)

    # ---------------- HRRR MSE ----------------
    sns.heatmap(
        pers_mse,
        ax=axes[0],
        cmap=grey_cmap,
        annot=True,
        fmt=".2f",
        cbar_kws={"label": "MSE"},
        linewidths=0.3,
        linecolor="white",
    )

    axes[0].set_title("HRRR (Persistence) MSE")
    axes[0].set_ylabel("Station")
    axes[0].set_xlabel("")

    # ---------------- Hybrid (annotated by MSE, colored by ΔMSE) ----------------
    sns.heatmap(
        diff_mse,  # <-- colors come from ΔMSE
        ax=axes[1],
        cmap=diverging_cmap,
        center=0,
        vmin=-vmax,
        vmax=vmax,
        annot=hybrid_mse,  # <-- text is raw Hybrid MSE
        fmt=".2f",
        cbar_kws={"label": r"$\Delta$MSE (Hybrid − HRRR)"},
        linewidths=0.3,
        linecolor="white",
    )

    axes[1].set_title("Hybrid MSE (colored by ΔMSE vs HRRR)")
    axes[1].set_ylabel("Station")
    axes[1].set_xlabel("Forecast Hour")

    plt.tight_layout()
    plt.show()

In [ ]:
weatherBench_percent(
    pers_df,
    hybrid_df,
)

In [ ]:
weatherBench_raw(
    pers_df,
    hybrid_df,
)

# state view MSE

In [ ]:
clims = pd.read_csv("/home/aevans/nwp_bias/src/landtype/data/oksm.csv")
clim_divs = clims["Climate_division"].unique().tolist()

In [ ]:
error_path = "/home/aevans/nwp_bias/src/machine_learning/data/error_visuals"

error_dir = os.listdir(error_path)
error_dir = sorted(error_dir)

error_dir = [e for e in error_dir if e in clim_divs]

temp_master_ls = []
wind_master_ls = []
tp_master_ls = []

temp_df_ls = []
wind_df_ls = []
precip_df_ls = []

for d in error_dir:
    print(d)
    try:
        error_df1 = pd.read_parquet(
            f"{error_path}/{d}/{d}_tp_error_metrics_master_normalized.parquet"
        )
        error_df2 = pd.read_parquet(
            f"{error_path}/{d}/{d}_u_total_error_metrics_master_normalized.parquet"
        )
        error_df3 = pd.read_parquet(
            f"{error_path}/{d}/{d}_t2m_error_metrics_master_normalized.parquet"
        )
        # Group by forecast hour (fh) and compute the mean MAE
        mean_mae_by_fh1 = error_df1.groupby("fh")["mae"].mean().reset_index()
        mean_mae_by_fh2 = error_df2.groupby("fh")["mae"].mean().reset_index()
        mean_mae_by_fh3 = error_df3.groupby("fh")["mae"].mean().reset_index()

        temp_df_ls.append(error_df3.reset_index())
        wind_df_ls.append(error_df2.reset_index())
        precip_df_ls.append(error_df1.reset_index())

        mae1 = mean_mae_by_fh1["mae"].values
        mae2 = mean_mae_by_fh2["mae"].values
        mae3 = mean_mae_by_fh3["mae"].values

        temp_master_ls.append(mae3)
        wind_master_ls.append(mae2)
        tp_master_ls.append(mae1)
    except:
        continue

In [ ]:
grouped = mae_df.groupby("station")["mae"].mean()
grouped

In [ ]:
nysm_df = pd.read_csv("/home/aevans/nwp_bias/src/landtype/data/nysm.csv")
print(nysm_df.columns)
keys = nysm_df["stid"].values
lats = nysm_df["lat [degrees]"].values
lons = nysm_df["lon [degrees]"].values
elevs = nysm_df["elevation [m]"].values

station_coords = {k: (lat, lon) for k, lat, lon in zip(keys, lats, lons)}
# elevations = {k: e for k, e in zip(keys, elevs)}

In [ ]:
grouped_df = pd.DataFrame({"station": grouped.index, "mae": grouped.values})

# Add lat/lon from your station_coords dictionary
grouped_df["lat_lon"] = grouped_df["station"].map(station_coords)
# grouped_df["elev"] = grouped_df["station"].map(elevations)

In [ ]:
grouped_df[["lat", "lon"]] = pd.DataFrame(
    grouped_df["lat_lon"].tolist(), index=grouped_df.index
)

In [ ]:
clim_div = [
    "St. Lawrence Valley",
    "Great Lakes",
    "Northern Plateau",
    "Champlain Valley",
    "Hudson Valley",
    "Mohawk Valley",
    "Western Plateau",
    "Eastern Plateau",
    "Coastal",
    "Central Lakes",
]
# # # clim_div = sorted(clim_div)
image = "/home/aevans/nwp_bias/src/landtype/data/NCEI_logo.png"
nysm_clim = pd.read_csv("/home/aevans/nwp_bias/src/landtype/data/nysm.csv")

In [ ]:
def create_state_mae(grouped_df, clim_div=clim_div, nysm_clim=nysm_clim, logo=image):
    # Create your dataframe df_
    df_ = nysm_clim.copy()
    font_size = 22

    # Create plot
    fig = plt.figure(figsize=(24, 16))
    ax = fig.add_subplot(
        1,
        1,
        1,
        projection=crs.LambertConformal(
            central_longitude=-75.0, standard_parallels=(49, 77)
        ),
    )

    # Load the shapefile for boundaries
    shapefile_path = "/home/aevans/nwp_bias/src/machine_learning/notebooks/data/GIS.OFFICIAL_CLIM_DIVISIONS.shp"
    gdf = gpd.read_file(shapefile_path)

    ny_state_boundaries_path = "/home/aevans/nwp_bias/src/landtype/data/State.shx"
    ny_state_boundaries_geo = gpd.read_file(ny_state_boundaries_path).to_crs(epsg=4326)

    ny_bbox = ny_state_boundaries_geo.total_bounds
    gdf_filtered = gdf.cx[ny_bbox[0] : ny_bbox[2], ny_bbox[1] : ny_bbox[3]]
    gdf_filtered = pd.concat([gdf_filtered.iloc[20:29], gdf_filtered.iloc[[32]]])

    # Create a categorical column for plotting
    gdf_filtered["category"] = np.arange(len(gdf_filtered))

    # Plot shapefile with climate divisions (remove the automatic legend)
    gdf_filtered.plot(
        ax=ax,
        transform=crs.PlateCarree(),
        column="category",
        cmap="tab10",
        alpha=0.3,
        legend=False,
    )

    # Create legend for climate divisions using the colors from the 'tab10' colormap and labels from 'clim_div'
    division_patches = [
        mpatches.Patch(
            color=plt.cm.tab10(i / len(gdf_filtered)), alpha=0.3, label=clim_div[i]
        )
        for i in range(len(gdf_filtered))
    ]

    # Add the climate divisions legend
    legend1 = ax.legend(
        handles=division_patches,
        loc="lower left",
        title="Climate Divisions",
        fontsize=font_size,
        title_fontsize=font_size,
    )
    ax.add_artist(legend1)  # Ensure the first legend is added to the plot

    # Set extent for the plot
    ax.set_extent([-80.0, -72.0, 40.0, 45.5], crs=crs.PlateCarree())

    # Add features
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), zorder=1)
    ax.gridlines(
        crs=crs.PlateCarree(),
        draw_labels=True,
        linewidth=2,
        color="black",
        alpha=0.5,
        linestyle="--",
    )

    # Normalize MAE for visual scaling
    mae = grouped_df["mae"].values
    min_mae, max_mae = mae.min(), mae.max()
    size_scaled = 300 + 1200 * (mae - min_mae) / (
        max_mae - min_mae
    )  # make size dynamic

    # Plot scatter points
    sc = ax.scatter(
        grouped_df["lon"],
        grouped_df["lat"],
        s=size_scaled,
        c=grouped_df["mae"],
        cmap="gist_stern",
        edgecolor="black",
        transform=crs.PlateCarree(),
        zorder=10,
        vmin=0.0,
        vmax=1.0,
    )

    # Annotate scatter points
    for i, row in grouped_df.iterrows():
        ax.annotate(
            row["station"],
            (row["lon"], row["lat"]),
            textcoords="offset points",
            xytext=(0, 15),
            ha="center",
            fontsize=15,
            color="black",
            transform=crs.PlateCarree(),
            zorder=20,
        )

    # Add colorbar
    cbar = plt.colorbar(sc, ax=ax, orientation="vertical", shrink=0.6)
    cbar.ax.tick_params(labelsize=font_size)
    cbar.set_label(r"MAE (°C)", fontsize=font_size)
    # (mm hr$^{-1}$) (°C)

    # Title and ticks
    plt.title(
        f"NYSM: Temperature-Error Predictions\nLSTM MAE Averaged Across Forecast-Hours",
        fontsize=font_size,
    )
    ax.tick_params(axis="x", labelsize=font_size)
    ax.tick_params(axis="y", labelsize=font_size)

    plt.tight_layout()
    plt.show()

In [ ]:
create_state_mae(grouped_df)

# Time of day/year

In [ ]:
def plot_combined_heatmap_day(
    clim_div,
    base_path,
    hrrr_file="ALL_u_total_hourly_abs_error_hrrr.csv",
    pers_file="ALL_u_total_hourly_abs_error_persistence.csv",
    lstm_file="ALL_u_total_hourly_error.csv",
    title="Combined Error Heatmap",
):

    def load_and_pivot(file):
        all_data = []
        for div in clim_div:
            csv_path = os.path.join(base_path, div, file)
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                df["Hour"] = df["Hour"].str.slice(0, 2).astype(int)
                df.rename(columns={"Mean_Absolute_Error": "error"}, inplace=True)
                df["Division"] = div
                all_data.append(df)
            else:
                print(f"Missing file: {csv_path}")

        df_all = pd.concat(all_data, ignore_index=True)

        heat = df_all.pivot_table(
            index="Division", columns="Hour", values="error", aggfunc="mean"
        )
        return heat.loc[clim_div]

    # Load
    hrrr_heat = load_and_pivot(hrrr_file)
    pers_heat = load_and_pivot(pers_file)
    lstm_heat = load_and_pivot(lstm_file)

    station_counts = nysm_df.groupby("Climate_division")["stid"].nunique().to_dict()

    # Build rows
    weights = np.array([station_counts[div] for div in clim_div])
    weights = weights / weights.sum()  # normalize to sum to 1

    # Weighted averages
    hrrr_row = np.average(hrrr_heat, axis=0, weights=weights)
    pers_row = np.average(pers_heat, axis=0, weights=weights)
    lstm_row = np.average(lstm_heat, axis=0, weights=weights)

    # Convert back to Series for consistent indexing
    hrrr_row = pd.Series(hrrr_row, index=hrrr_heat.columns)
    pers_row = pd.Series(pers_row, index=pers_heat.columns)
    lstm_row = pd.Series(lstm_row, index=lstm_heat.columns)

    diff_lstm_row = hrrr_row - pers_row
    diff_lstm_row_perc = (diff_lstm_row / hrrr_row) * 100

    top = pd.DataFrame(
        [hrrr_row, pers_row, diff_lstm_row_perc],
        index=["HRRR (MAE)", "LSTM (MAE)", "LSTM % Improvement"],
    )

    # Add name to LSTM row
    lstm_row.name = "All Stations"

    # Build final table
    final = pd.concat([top, lstm_heat, lstm_row.to_frame().T], axis=0)

    ##################################################################
    # SINGLE HEATMAP TRICK: we mask each row group for different cmap
    ##################################################################

    mask_grey = np.ones_like(final, dtype=bool)  # Row 0
    mask_row2 = np.ones_like(final, dtype=bool)  # Row 1 (unique cmap)
    mask_diff = np.ones_like(final, dtype=bool)  # Row 2 (difference)
    mask_main = np.ones_like(final, dtype=bool)  # Rows 3+

    mask_grey[0] = False  # Only row 0
    mask_row2[1] = False  # Only row 1
    mask_diff[2] = False  # Only row 2
    mask_main[3:] = False  # All remaining

    fig, ax = plt.subplots(figsize=(26, 12))

    # -------------------------------------------------------------
    # 1. MAIN PiYG heatmap (bottom layer)
    # -------------------------------------------------------------
    hm_main = sns.heatmap(
        final,
        cmap="PiYG",
        mask=mask_main,
        linewidths=0.4,
        ax=ax,
        cbar=False,
        vmin=-1,
        vmax=1,
    )

    # Grab the *correct* PiYG QuadMesh
    quad_piyg = hm_main.collections[0]
    # -------------------------------------------------------------
    # 4. RIGHT PiYG colorbar (main errors)
    # -------------------------------------------------------------
    cbar_right = plt.colorbar(quad_piyg, ax=ax, location="right", pad=0.02, shrink=0.75)
    cbar_right.set_label(r"Mean Error (m s$^{-1}$)", fontsize=16)

    # (m s$^{-1}$)
    # (°C)
    # -------------------------------------------------------------
    # 2. GREYS top row
    # -------------------------------------------------------------
    # Extract only the visible values in the GREYS row
    grey_values = final.iloc[0][~mask_grey[0]]

    # Compute dynamic bounds
    vmin_grey = grey_values.min()
    vmax_grey = grey_values.max()

    sns.heatmap(
        final,
        cmap="Greys",
        mask=mask_grey,
        ax=ax,
        cbar=False,
        vmin=vmin_grey,
        vmax=vmax_grey,
    )

    # -------------------------------------------------------------
    # 2B. ROW 1: Persistence values, but colored using % difference
    # -------------------------------------------------------------
    # Build temp matrix full of NaNs
    temp = final.copy() * np.nan

    # Put only diff_lstm_row into row 1 (color will come from this)
    temp.iloc[1] = diff_lstm_row.values

    # Draw heatmap for ROW 1 but colored by difference
    sns.heatmap(
        temp,
        cmap="RdBu",
        mask=mask_row2,  # mask everything except row 1
        center=0,
        ax=ax,
        cbar=False,
        vmin=-1,
        vmax=1,
    )

    # -------------------------------------------------------------
    # 3. DIFFERENCE RdBu heatmap (top of PiYG except row 0)
    # -------------------------------------------------------------
    hm_diff = sns.heatmap(
        final,
        cmap="RdBu",
        mask=mask_diff,
        center=0,
        ax=ax,
        cbar=True,
        vmin=-50,
        vmax=50,
        cbar_kws={
            "orientation": "vertical",
            "location": "left",
            "pad": 0.2,
            "label": "← Worse | % Diff. b/w Persistence and LSTM | Better →",
            "fraction": 0.01,  # shrink colorbar (smaller = shorter)
            "aspect": 30,  # optional: change thickness
        },
    )
    # >>> ADD THESE LINES <<<
    hm_diff.collections[0].colorbar.ax.tick_params(labelsize=16)
    hm_diff.collections[0].colorbar.ax.yaxis.label.set_size(16)

    # Add an emphasized line between row 2 and the rest (i.e., between row index 1 and 2)
    ax.hlines(
        y=3,  # boundary between second and third row
        xmin=0,
        xmax=final.shape[1],  # number of columns
        color="black",
        linewidth=3,
    )

    for i in range(final.shape[0]):
        for j in range(final.shape[1]):

            if not (
                mask_grey[i, j]
                and mask_row2[i, j]
                and mask_diff[i, j]
                and mask_main[i, j]
            ):

                val = final.values[i, j]
                if pd.notna(val):

                    # --------------------------------------------------
                    # Choose correct colormap + normalization per row
                    # --------------------------------------------------
                    if i == 0:
                        cmap = plt.get_cmap("Greys")
                        norm = mcolors.Normalize(vmin=vmin_grey, vmax=vmax_grey)
                        val_for_color = val

                    elif i == 1:
                        cmap = plt.get_cmap("RdBu")
                        norm = mcolors.Normalize(vmin=-1, vmax=1)
                        val_for_color = diff_lstm_row.iloc[j]

                    elif i == 2:
                        cmap = plt.get_cmap("RdBu")
                        norm = mcolors.Normalize(vmin=-50, vmax=50)
                        val_for_color = val

                    else:
                        cmap = plt.get_cmap("PiYG")
                        norm = mcolors.Normalize(vmin=-1, vmax=1)
                        val_for_color = val

                    rgba = cmap(norm(val_for_color))
                    r, g, b = rgba[:3]

                    luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
                    text_color = "white" if luminance < 0.5 else "black"

                    ax.text(
                        j + 0.5,
                        i + 0.5,
                        f"{val:.2f}",
                        ha="center",
                        va="center",
                        fontsize=12,
                        color=text_color,
                        zorder=20,
                    )

    plt.title(title, fontsize=22)
    plt.xlabel("Local Hour of Day", fontsize=18)
    plt.ylabel("Climate Divisions", fontsize=18)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16, rotation=0)

    plt.tight_layout()
    plt.show()

    return final

In [ ]:
final_df = plot_combined_heatmap(
    clim_div=clim_div,
    base_path="/home/aevans/nwp_bias/src/machine_learning/data/error_visuals/dataframes",
    title="NYSM: Temperature Error Predictions\n LSTM Error Grouped By Local Time of Day",
)

In [ ]:
def plot_combined_heatmap_month(
    clim_div,
    base_path,
    title,
    hrrr_file="ALL_tp_monthly_error_abs_hrrr.csv",
    pers_file="ALL_tp_monthly_error_abs_persistence.csv",
    lstm_file="ALL_tp_monthly_error_abs.csv",
):

    # -------------------------------------------------------------
    # FUNCTION TO LOAD EACH MODEL'S CSV AND PIVOT TO DIV x MONTH
    # -------------------------------------------------------------
    def load_and_pivot(file):
        all_data = []
        for div in clim_div:
            csv_path = os.path.join(base_path, div, file)

            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)

                # Ensure month is integer (if string)
                df["Month"] = pd.to_datetime(df["Month"], format="%B").dt.month

                df.rename(columns={"Mean_Error": "error"}, inplace=True)
                df["Division"] = div

                all_data.append(df)
            else:
                print(f"Missing file: {csv_path}")

        df_all = pd.concat(all_data, ignore_index=True)

        heat = df_all.pivot_table(
            index="Division", columns="Month", values="error", aggfunc="mean"
        )

        return heat.loc[clim_div]

    # -------------------------------------------------------------
    # LOAD HRRR, Persistence, LSTM — NOW BY MONTH
    # -------------------------------------------------------------
    hrrr_heat = load_and_pivot(hrrr_file)
    pers_heat = load_and_pivot(pers_file)
    lstm_heat = load_and_pivot(lstm_file)

    # -------------------------------------------------------------
    # STATION WEIGHTS FOR DIVISIONAL AVERAGE
    # -------------------------------------------------------------
    station_counts = (
        nysm_df.groupby("climate_division_name")["stid"].nunique().to_dict()
    )

    weights = np.array([station_counts[div] for div in clim_div])
    weights = weights / weights.sum()

    # Weighted divisional mean rows
    hrrr_row = pd.Series(
        np.average(hrrr_heat, axis=0, weights=weights), index=hrrr_heat.columns
    )
    pers_row = pd.Series(
        np.average(pers_heat, axis=0, weights=weights), index=pers_heat.columns
    )
    lstm_row = pd.Series(
        np.average(lstm_heat, axis=0, weights=weights), index=lstm_heat.columns
    )

    diff_lstm_row = hrrr_row - pers_row
    diff_lstm_row_perc = (diff_lstm_row / hrrr_row) * 100

    top = pd.DataFrame(
        [hrrr_row, pers_row, diff_lstm_row_perc],
        index=["HRRR (MAE)", "LSTM (MAE)", "LSTM % Improvement"],
    )

    lstm_row.name = "All Stations"

    # -------------------------------------------------------------
    # BUILD FINAL TABLE (2 special rows + divisions + summary row)
    # -------------------------------------------------------------
    final = pd.concat([top, lstm_heat, lstm_row.to_frame().T], axis=0)

    # -------------------------------------------------------------
    # MASKS FOR MULTI-LAYER HEATMAP
    # -------------------------------------------------------------
    mask_grey = np.ones_like(final, dtype=bool)  # Row 0
    mask_row2 = np.ones_like(final, dtype=bool)  # Row 1 (unique cmap)
    mask_diff = np.ones_like(final, dtype=bool)  # Row 2 (difference)
    mask_main = np.ones_like(final, dtype=bool)  # Rows 3+

    mask_grey[0] = False  # Only row 0
    mask_row2[1] = False  # Only row 1
    mask_diff[2] = False  # Only row 2
    mask_main[3:] = False  # All remaining

    fig, ax = plt.subplots(figsize=(26, 12))

    # -------------------------------------------------------------
    # MAIN PiYG LAYER
    # -------------------------------------------------------------
    hm_main = sns.heatmap(
        final,
        cmap="Purples",
        mask=mask_main,
        linewidths=0.4,
        ax=ax,
        cbar=False,
        vmin=0,
        vmax=12,
    )

    quad_piyg = hm_main.collections[0]

    cbar_right = plt.colorbar(quad_piyg, ax=ax, location="right", pad=0.02, shrink=0.75)
    cbar_right.set_label(r"Mean Absolute Error (mm hr$^{-1}$)", fontsize=16)

    # -------------------------------------------------------------
    # GREYS ROW – dynamic vmin/vmax
    # -------------------------------------------------------------
    grey_values = final.iloc[0][~mask_grey[0]]

    vmin_grey = grey_values.min() - 0.35
    vmax_grey = grey_values.max()

    sns.heatmap(
        final,
        cmap="Greys",
        mask=mask_grey,
        ax=ax,
        cbar=False,
        vmin=vmin_grey,
        vmax=vmax_grey,
    )

    # -------------------------------------------------------------
    # 2B. ROW 1: Persistence values, but colored using % difference
    # -------------------------------------------------------------
    # Build temp matrix full of NaNs
    temp = final.copy() * np.nan

    # Put only diff_lstm_row into row 1 (color will come from this)
    temp.iloc[1] = diff_lstm_row.values

    # Draw heatmap for ROW 1 but colored by difference
    sns.heatmap(
        temp,
        cmap="RdBu",
        mask=mask_row2,  # mask everything except row 1
        center=0,
        ax=ax,
        cbar=False,
        vmin=-1,
        vmax=1,
    )

    # -------------------------------------------------------------
    # RdBu DIFFERENCE LAYER
    # -------------------------------------------------------------
    hm_diff = sns.heatmap(
        final,
        cmap="RdBu",
        mask=mask_diff,
        center=0,
        ax=ax,
        cbar=True,
        vmin=-50,
        vmax=50,
        cbar_kws={
            "orientation": "vertical",
            "location": "left",
            "pad": 0.2,
            "label": "← Worse | % Diff. b/w Persistence and LSTM | Better →",
            "fraction": 0.01,
            "aspect": 30,
        },
    )

    hm_diff.collections[0].colorbar.ax.tick_params(labelsize=16)
    hm_diff.collections[0].colorbar.ax.yaxis.label.set_size(16)

    # -------------------------------------------------------------
    # HORIZONTAL LINE SEPARATING SPECIAL ROWS
    # -------------------------------------------------------------
    ax.hlines(y=3, xmin=0, xmax=final.shape[1], color="black", linewidth=3)

    # -------------------------------------------------------------
    # ANNOTATE CELLS
    # -------------------------------------------------------------

    for i in range(final.shape[0]):
        for j in range(final.shape[1]):

            if not (
                mask_grey[i, j]
                and mask_row2[i, j]
                and mask_diff[i, j]
                and mask_main[i, j]
            ):

                val = final.values[i, j]
                if pd.notna(val):

                    # --------------------------------------------------
                    # Choose correct colormap + normalization per row
                    # --------------------------------------------------
                    if i == 0:
                        cmap = plt.get_cmap("Greys")
                        norm = mcolors.Normalize(vmin=vmin_grey, vmax=vmax_grey)
                        val_for_color = val

                    elif i == 1:
                        cmap = plt.get_cmap("RdBu")
                        norm = mcolors.Normalize(vmin=-1, vmax=1)
                        val_for_color = diff_lstm_row.iloc[j]

                    elif i == 2:
                        cmap = plt.get_cmap("RdBu")
                        norm = mcolors.Normalize(vmin=-50, vmax=50)
                        val_for_color = val

                    else:
                        cmap = plt.get_cmap("Purples")
                        norm = mcolors.Normalize(vmin=0, vmax=12)
                        val_for_color = val

                    rgba = cmap(norm(val_for_color))
                    r, g, b = rgba[:3]

                    luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
                    text_color = "white" if luminance < 0.5 else "black"

                    ax.text(
                        j + 0.5,
                        i + 0.5,
                        f"{val:.2f}",
                        ha="center",
                        va="center",
                        fontsize=12,
                        color=text_color,
                        zorder=20,
                    )

    plt.title(title, fontsize=22)
    plt.xlabel("Month of Year", fontsize=18)
    plt.ylabel("Climate Divisions", fontsize=18)
    plt.xticks(
        ticks=np.arange(12) + 0.5,
        labels=[
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
        ],
        fontsize=16,
    )
    plt.yticks(fontsize=16, rotation=0)

    plt.tight_layout()
    plt.show()

    return final

In [ ]:
final_df = plot_combined_heatmap_month(
    clim_div=clim_div,
    base_path="/home/aevans/nwp_bias/src/machine_learning/src/visuals/dataframes",
    title="NYSM: Precipitation Error Predictions\n LSTM MAE Grouped By Month",
)